In [1]:
import sqlite3
import pandas as pd

# 1. Load clean data and connect to a local SQLite database
df = pd.read_csv('../data/clean_ecommerce_data.csv')
conn = sqlite3.connect('../sql/ecommerce.db')

# 2. Push the DataFrame into a real SQL table named 'orders'
df.to_sql('orders', conn, if_exists='replace', index=False)

# 3. Helper function to run SQL queries cleanly
def run_query(sql):
    return pd.read_sql_query(sql, conn)

print("SQL database 'ecommerce.db' ready! Total rows:", run_query("SELECT COUNT(*) AS total_rows FROM orders")['total_rows'][0])


SQL database 'ecommerce.db' ready! Total rows: 9994


In [2]:
q1 = """
SELECT 
    COUNT(DISTINCT order_id) AS total_orders,
    COUNT(DISTINCT customer_id) AS total_customers,
    ROUND(SUM(sales), 2) AS total_revenue,
    ROUND(SUM(profit), 2) AS total_profit,
    ROUND((SUM(profit) / SUM(sales)) * 100, 2) AS overall_margin_pct
FROM orders;
"""
run_query(q1)


,total_orders,total_customers,total_revenue,total_profit,overall_margin_pct
0,5009,793,2297200.86,286397.02,12.47


In [3]:
q2 = """
SELECT 
    CASE 
        WHEN discount = 0 THEN '1. No Discount (0%)'
        WHEN discount <= 0.20 THEN '2. Safe Discount (1-20%)'
        ELSE '3. Heavy Discount (>20%)'
    END AS discount_tier,
    COUNT(*) AS order_lines,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit,
    ROUND(AVG(profit), 2) AS avg_profit_per_item
FROM orders
GROUP BY discount_tier
ORDER BY discount_tier;
"""
run_query(q2)


,discount_tier,order_lines,total_sales,total_profit,avg_profit_per_item
0,1. No Discount (0%),4798,1087908.47,320987.60,66.90
1,2. Safe Discount (1-20%),3803,846522.24,100785.47,26.50
2,3. Heavy Discount (>20%),1393,362770.15,-135376.06,-97.18


In [4]:
q3 = """
SELECT 
    category,
    sub_category,
    product_name,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_loss
FROM orders
GROUP BY category, sub_category, product_name
HAVING SUM(profit) < -500
ORDER BY total_loss ASC
LIMIT 10;
"""
run_query(q3)


,category,sub_category,product_name,total_sales,total_loss
0,Technology,Machines,Cubify CubeX 3D Printer Double Head Print,11099.96,-8879.97
1,Technology,Machines,Lexmark MX611dhe Monochrome Laser Printer,16829.90,-4589.97
2,Technology,Machines,Cubify CubeX 3D Printer Triple Head Print,7999.98,-3839.99
3,Furniture,Tables,Chromcraft Bull-Nose Wood Oval Conference Tabl...,9917.64,-2876.12
4,Furniture,Tables,Bush Advantage Collection Racetrack Conference...,9544.73,-1934.40
5,Office Supplies,Binders,GBC DocuBind P400 Electric Binding System,17965.07,-1878.17
6,Technology,Machines,Cisco TelePresence System EX90 Videoconferenci...,22638.48,-1811.08
7,Office Supplies,Supplies,Martin Yale Chadless Opener Electric Letter Op...,16656.20,-1299.18
8,Furniture,Tables,Balt Solid Wood Round Tables,6518.75,-1201.06
9,Furniture,Tables,BoxOffice By Design Rectangular and Half-Moon ...,1706.25,-1148.44


In [5]:
q4 = """
WITH regional_subcats AS (
    SELECT 
        region,
        sub_category,
        ROUND(SUM(profit), 2) AS total_profit,
        RANK() OVER (PARTITION BY region ORDER BY SUM(profit) DESC) AS profit_rank
    FROM orders
    GROUP BY region, sub_category
)
SELECT * 
FROM regional_subcats
WHERE profit_rank <= 3;
"""
run_query(q4)


,region,sub_category,total_profit,profit_rank
0,Central,Copiers,15608.84,1
1,Central,Phones,12323.03,2
2,Central,Accessories,7251.63,3
3,East,Copiers,17022.84,1
4,East,Phones,12314.69,2
5,East,Binders,11267.93,3
6,South,Phones,10767.28,1
7,South,Accessories,7004.54,2
8,South,Chairs,6612.09,3
9,West,Copiers,19327.24,1


In [6]:
q5 = """
SELECT 
    ship_mode,
    COUNT(DISTINCT order_id) AS total_orders,
    ROUND(AVG(shipping_days), 1) AS avg_delivery_days,
    ROUND(SUM(profit), 2) AS total_profit
FROM orders
GROUP BY ship_mode
ORDER BY avg_delivery_days ASC;
"""
run_query(q5)


,ship_mode,total_orders,avg_delivery_days,total_profit
0,Same Day,264,0.0,15891.76
1,First Class,787,2.2,48969.84
2,Second Class,964,3.2,57446.64
3,Standard Class,2994,5.0,164088.79


In [7]:
q6 = """
SELECT 
    customer_id,
    customer_name,
    segment,
    COUNT(DISTINCT order_id) AS orders_placed,
    ROUND(SUM(sales), 2) AS lifetime_sales,
    ROUND(SUM(profit), 2) AS lifetime_profit
FROM orders
GROUP BY customer_id, customer_name, segment
ORDER BY lifetime_sales DESC
LIMIT 5;
"""
run_query(q6)


,customer_id,customer_name,segment,orders_placed,lifetime_sales,lifetime_profit
0,SM-20320,Sean Miller,Home Office,5,25043.05,-1980.74
1,TC-20980,Tamara Chand,Corporate,5,19052.22,8981.32
2,RB-19360,Raymond Buch,Consumer,6,15117.34,6976.10
3,TA-21385,Tom Ashbrook,Home Office,4,14595.62,4703.79
4,AB-10105,Adrian Barton,Consumer,10,14473.57,5444.81


### SQL Database Modeling & Business Querying (SQLite)

**1. Relational Database Ingestion (`ecommerce.db`)**
* Loaded the cleaned 9,994-row transactional dataset (`clean_ecommerce_data.csv`) into a relational **SQLite** database (`orders` table) using Python's `sqlite3` and `pandas` libraries.
* Built a reusable `run_query()` helper wrapper to execute ad-hoc SQL queries and output structured business reports.

**2. Core Corporate SQL Queries & Findings**
* **Executive KPI Summary (`COUNT DISTINCT`, `SUM`, `ROUND`):** Calculated company-wide performance across **5,009 unique orders** and **793 unique customers**, generating **`$2,297,200.86` in total revenue**, **`$286,397.02` in net profit**, and an overall **profit margin of `12.47%`**.
* **Discount Tier Segmentation (`CASE WHEN` + `GROUP BY`):** Segmented all order lines into three pricing buckets (`No Discount`, `Safe Discount 1–20%`, and `Heavy Discount >20%`). Confirmed that orders with **0%–20% discounts generated over `$421,000` in combined profit**, whereas **heavy discounts (>20%) caused a net loss of `-$135,376.06`** across 1,393 order lines.
* **Worst Money-Losing Products (`GROUP BY` + `HAVING`):** Applied a post-aggregation materiality filter (`HAVING SUM(profit) < -500`) to isolate the top 10 worst-performing SKUs led by **`Cubify CubeX 3D Printer Double Head Print` (`-$8,879.97` net loss)**, giving procurement and pricing teams an immediate action list.
* **Regional Top-3 Sub-Categories (`WITH` CTE + `RANK()` Window Function):** Used `RANK() OVER (PARTITION BY region ORDER BY SUM(profit) DESC)` to rank sub-categories inside each region independently, identifying **Copiers**, **Phones**, **Accessories**, and **Paper** as core profit drivers across regions.
* **Fulfillment Speed vs. Profitability (`AVG` + `GROUP BY`):** Verified average delivery times across shipping tiers—**Same Day (`0.0` days)**, **First Class (`2.2` days)**, **Second Class (`3.2` days)**, and **Standard Class (`5.0` days)**—confirming SLA adherence while showing Standard Class drives the highest volume and total profit (`$164,088.79`).
* **VIP Customer Retention (`GROUP BY` + `ORDER BY`):** Identified the top 5 accounts by lifetime revenue led by **Sean Miller (`$25,043.05` sales)**, while flagging that heavy discounting on VIP orders caused negative lifetime profit (`-$1,980.74`) on his account.
